# 02 · UN General Assembly Voting — Cold War Blocks vs. Natural Alignment

## Thesis

The Cold War superimposed a **binary frame** (West vs. East) on world politics.
PCA of UN General Assembly votes shows that:
- The true geopolitical space is **multi-dimensional**
- "Blocks" are approximations, not reality
- Nations have a **continuous, organic** alignment profile

The same principle as the congress analysis, applied to nation-states.


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import plotly.express as px

from voting_pca import (
    load_un_data,
    build_un_vote_matrix,
    run_pca,
    run_clustering,
    silhouette_analysis,
    plot_un_country_clusters,
    make_un_interactive,
    BLOCK_COLORS,
)

OUT_DIR = PROJECT_ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)
print("Environment ready.")


## 1 · Load and Explore the Data

Data from the `unvotes` R package (mirrored on TidyTuesday).
Covers UNGA resolutions from **1946 to ~2019**.


In [ ]:
un_votes, un_rc, un_issues = load_un_data()
print("\nVote values:")
print(un_votes["vote"].value_counts())


In [ ]:
print("\nIssue categories:")
print(un_issues["issue"].value_counts())


## 2 · Build the Country × Resolution Matrix

Each row = a country.  Each column = a UNGA resolution (rcid).
+1 = yes, −1 = no, 0 = abstain / absent.


In [ ]:
un_matrix, blocks = build_un_vote_matrix(un_votes)
print("\nBlock assignments:")
print(blocks.value_counts())


## 3 · PCA

If the Cold War binary explained global alignment, PC1 would separate East
from West cleanly and explain ~100% of variance.


In [ ]:
un_pca_coords, un_pca_model, un_var = run_pca(un_matrix, n_components=10)
print("Variance explained:")
for i, v in enumerate(un_var):
    print(f"  PC{i+1}: {v:.1%}  (cum: {un_var[:i+1].sum():.1%})")


## 4 · Cluster the Countries

K-means with k=4 often recovers:
- Western bloc
- Eastern bloc
- Non-aligned / Global South
- A fourth, nuanced grouping

But the *boundaries* are fuzzy and many countries resist clean assignment.


In [ ]:
un_clusters = run_clustering(un_pca_coords, k_range=[2, 3, 4, 5, 6])


## 5 · Visualise: Cold War Blocks vs. Natural Clusters


In [ ]:
plot_un_country_clusters(
    un_pca_coords,
    un_matrix.index,
    blocks,
    cluster_labels=un_clusters[4],
)
from IPython.display import Image
Image(str(OUT_DIR / "un_country_clusters.png"))


In [ ]:
make_un_interactive(un_pca_coords, un_matrix.index, blocks)
print("Interactive UN plot → outputs/un_interactive.html")


## 6 · Which Countries Cross Blocks Most?

The Non-Aligned Movement was an explicit rejection of the imposed binary.
PCA places these countries in the *middle* of the spectrum,
exactly as the thesis predicts.


In [ ]:
pc1_scores = pd.Series(un_pca_coords[:, 0], index=un_matrix.index, name="PC1")
pc2_scores = pd.Series(un_pca_coords[:, 1], index=un_matrix.index, name="PC2")

alignment_df = pd.concat([pc1_scores, pc2_scores, blocks], axis=1)
print("\nNon-Aligned movement countries — PC1 scores (middle = non-binary):")
nonaligned = alignment_df[alignment_df["block"] == "Non-Aligned"].sort_values("PC1")
print(nonaligned[["PC1", "PC2"]].to_string())


## 7 · Silhouette: Is k=2 the Natural Answer?


In [ ]:
un_sil = silhouette_analysis(un_pca_coords, k_range=[2, 3, 4, 5, 6])
print("Silhouette scores:")
for k, s in sorted(un_sil.items()):
    mark = " ← best" if s == max(un_sil.values()) else ""
    print(f"  k={k}: {s:.4f}{mark}")


## 8 · Conclusion

The UNGA voting record shows:
- Nations do **not** split neatly into two camps
- The Non-Aligned Movement occupied genuine *middle ground* in PCA space
- Natural k-means clusters find **more than 2** meaningful groupings
- The Cold War "bloc" framing was a political construction imposed on
  a more complex and continuous reality

The same pattern as congress voting, the same pattern as individual belief:
**imposed binaries are a cognitive shortcut that erases real structure.**
